## Installing & Connection Phase

In [1]:
pip install sqlalchemy psycopg2-binary pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# !pip install psycopg2-binary pandas

import pandas as pd
import psycopg2

MY_LOGIN = 'ixojambetov'
MY_PASSWORD = 'Ixojambetov@Db9842'

conn = psycopg2.connect(
    host='thomas.proxy.rlwy.net',
    port=51432,
    dbname='academy_db',
    user=MY_LOGIN,
    password=MY_PASSWORD,
    sslmode='require'
)

def sql(query):
    """Выполнить SQL и вернуть результат как DataFrame."""
    return pd.read_sql(query, conn)

print('Подключено')

Подключено


## Этап 1: Профилирование и масштабы данных (Data Profiling)

**Описание:** На этом этапе оценивается физический объем базы данных и её временные рамки. Цель — понять размер выборки, с которой предстоит работать (общее количество строк в таблицах, число уникальных мерчантов и терминалов), а также определить период времени, за который логировались транзакции, прежде чем переходить к сложным финансовым расчетам.

### 1.1. Общее количество строк в таблицах

In [3]:
sql("""
SELECT 'transactions_1' AS table_name, COUNT(*) AS total_rows FROM public.ds_transactions_1
UNION ALL
SELECT 'settlements', COUNT(*) FROM public.ds_settlements
UNION ALL
SELECT 'disputes', COUNT(*) FROM public.ds_disputes
UNION ALL
SELECT 'merchants_1', COUNT(*) FROM public.ds_merchants_1
UNION ALL
SELECT 'terminals_1', COUNT(*) FROM public.ds_terminals_1;
""")

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,table_name,total_rows
0,merchants_1,1200
1,disputes,1986
2,terminals_1,2527
3,settlements,51052
4,transactions_1,80000


### 1.2. Временной диапазон датасета

In [4]:
sql("""
SELECT
    MIN(txn_ts) AS first_transaction_date,
    MAX(txn_ts) AS last_transaction_date,
    MAX(txn_ts) - MIN(txn_ts) AS total_time_span
FROM public.ds_transactions_1;
""")

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,first_transaction_date,last_transaction_date,total_time_span
0,2025-01-01 00:27:00,2025-12-28 23:49:00,361 days 23:22:00


### 1.3. Количество активных бизнес-сущностей

In [6]:
display(sql("""

SELECT
    COUNT(DISTINCT merchant_id) AS unique_merchants_count,
    COUNT(DISTINCT terminal_id) AS unique_terminals_count
FROM public.ds_terminals_1;

"""))

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,unique_merchants_count,unique_terminals_count
0,1200,2527


## Этап 2: Описательная статистика (Descriptive Statistics)

**Описание:** Этот шаг фокусируется на агрегированных финансовых показателях эквайринга. Здесь рассчитываются глобальные денежные потоки (совокупный оборот, заработанная эквайером комиссия, чистые выплаты мерчантам) и метрики «чека» (минимальная, максимальная, средняя и медианная суммы транзакций). Это дает верхнеуровневое представление о реальной экономике системы.

### 2.1. Глобальные финансовые итоги эквайринга

In [9]:
sql("""
SELECT
    ROUND(SUM(gross_amount_uzs)) AS total_gross_turnover,
    ROUND(SUM(commission_uzs)) AS total_commission_earned,
    ROUND(SUM(net_amount_uzs)) AS total_net_payout,
    ROUND(AVG(commission_uzs::numeric / NULLIF(gross_amount_uzs, 0) * 100), 2) AS effective_avg_commission_percent
FROM public.ds_settlements;
""")

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,total_gross_turnover,total_commission_earned,total_net_payout,effective_avg_commission_percent
0,2.327253e+10,346042460.0,2.230461e+10,1.15


### 2.2. Анализ чека успешных транзакций

In [11]:
sql("""
SELECT
    MIN(amount_uzs) AS minimum_check,
    MAX(amount_uzs) AS maximum_check,
    ROUND(AVG(amount_uzs), 2) AS average_check,
    percentile_cont(0.5) WITHIN GROUP (ORDER BY amount_uzs) AS median_check
FROM public.ds_transactions_1
WHERE status = 'approved';
""")

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,minimum_check,maximum_check,average_check,median_check
0,3800,16139300,320492.65,177900.0


## Этап 3: Структурная сегментация (Data Slicing)

**Описание:** Данный этап подразумевает разделение общей массы транзакций на логические бизнес-сегменты для выявления точек концентрации капитала. Данные разрезаются и группируются по категориальным признакам (доли платежных систем, соотношение POS-терминалов и E-commerce, топ регионов), чтобы в процентном выражении увидеть, кто именно формирует основу бизнеса.

### 3.1. Доли платежных систем (Uzcard, Humo и др.)

In [14]:
sql(
"""
SELECT
    card_scheme,
    COUNT(*) AS transaction_count,
    SUM(amount_uzs) AS total_turnover_uzs,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS transaction_share_percent,
    ROUND(SUM(amount_uzs) * 100.0 / SUM(SUM(amount_uzs)) OVER(), 2) AS volume_share_percent
FROM public.ds_transactions_1
WHERE status = 'approved'
GROUP BY card_scheme
ORDER BY total_turnover_uzs DESC;
"""
)

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,card_scheme,transaction_count,total_turnover_uzs,transaction_share_percent,volume_share_percent
0,UZCARD,34159,1.005521e+10,46.50,42.71
1,HUMO,23697,7.041906e+09,32.26,29.91
2,VISA,9893,3.929585e+09,13.47,16.69
3,MASTERCARD,5716,2.518290e+09,7.78,10.70


### 3.2. Распределение оборота по типам интерфейсов (POS vs E-commerce)

In [16]:
sql("""
SELECT
    term.terminal_type,
    COUNT(t.txn_id) AS transaction_count,
    SUM(t.amount_uzs) AS total_turnover_uzs,
    ROUND(SUM(t.amount_uzs) * 100.0 / SUM(SUM(t.amount_uzs)) OVER(), 2) AS volume_share_percent
FROM public.ds_transactions_1 t
JOIN public.ds_terminals_1 term ON t.terminal_id = term.terminal_id
WHERE t.status = 'approved'
GROUP BY term.terminal_type
ORDER BY total_turnover_uzs DESC;
""")

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,terminal_type,transaction_count,total_turnover_uzs,volume_share_percent
0,POS,57304,1.661378e+10,70.56
1,ECOM,7160,4.580232e+09,19.45
2,QR,9001,2.350979e+09,9.99


### 3.3. Географический срез: Топ-5 регионов

In [17]:
sql("""
SELECT
    r.region_name,
    COUNT(t.txn_id) AS transaction_count,
    SUM(t.amount_uzs) AS regional_turnover_uzs,
    ROUND(SUM(t.amount_uzs) * 100.0 / SUM(SUM(t.amount_uzs)) OVER(), 2) AS regional_share_percent
FROM public.ds_transactions_1 t
JOIN public.ds_terminals_1 term ON t.terminal_id = term.terminal_id
JOIN public.ds_regions r ON term.region_id = r.region_id
WHERE t.status = 'approved'
GROUP BY r.region_name
ORDER BY regional_turnover_uzs DESC
LIMIT 5;
""")

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,region_name,transaction_count,regional_turnover_uzs,regional_share_percent
0,Toshkent shahri,18474,4.221085e+09,17.93
1,Samarqand,7033,2.520012e+09,10.70
2,Xorazm,4122,2.232144e+09,9.48
3,Andijon,6700,1.840597e+09,7.82
4,Jizzax,3015,1.642991e+09,6.98


## Этап 4: Анализ аномалий и качества данных (Data Quality)

**Описание:** Заключительный этап базового обзора направлен на выявление операционных рисков и технических проблем в массиве данных. Здесь анализируется уровень отказов (доля неуспешных транзакций) и подробно изучается статистика по спорам, включая рейтинг самых частых причин чарджбэков и общую конверсию проблемных платежей, что напрямую отвечает на запрос заказчика о финансовых потерях.

### 4.1. Анализ распределения статусов транзакций

In [18]:
sql("""
SELECT
    status,
    COUNT(*) AS transaction_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS share_percent
FROM public.ds_transactions_1
GROUP BY status
ORDER BY transaction_count DESC;
""")

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,status,transaction_count,share_percent
0,approved,73465,91.83
1,declined,5713,7.14
2,reversed,822,1.03


### 4.2. Рейтинг причин открытия споров (Чарджбэков)

In [ ]:
sql("""
SELECT
    reason_code,
    COUNT(*) AS dispute_count,
    SUM(dispute_amount_uzs) AS total_dispute_volume_uzs,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS dispute_reason_share_percent
FROM public.ds_disputes
GROUP BY reason_code
ORDER BY dispute_count DESC;
""")

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,reason_code,dispute_count,total_dispute_volume_uzs,dispute_reason_share_percent
0,product_not_received,723,171938283.0,36.40
1,quality,542,146474351.0,27.29
2,subscription_cancel,269,77151328.0,13.54
3,duplicate,252,72969928.0,12.69
4,fraud,200,208717467.0,10.07


### 4.3. Конверсия транзакций в споры (Dispute Rate)

In [ ]:
sql("""
SELECT
    COUNT(t.txn_id) AS total_transactions,
    COUNT(d.dispute_id) AS total_disputes,
    ROUND(COUNT(d.dispute_id) * 100.0 / COUNT(t.txn_id), 4) AS overall_dispute_rate_percent,
    SUM(d.dispute_amount_uzs) AS total_money_at_risk_uzs
FROM public.ds_transactions_1 t
LEFT JOIN public.ds_disputes d ON t.txn_id = d.txn_id;
""")

/var/folders/7r/l55tlv913fd40895kn_wpch80000gn/T/ipykernel_33498/1627210578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,total_transactions,total_disputes,overall_dispute_rate_percent,total_money_at_risk_uzs
0,80000,1986,2.4825,677251357.0
